# Bay Area Transit Equity — Interactive Visualization
**Group 13 · FY2025 Analysis**

This notebook produces a two-pane interactive visualization:
- **Left**: Zoomable map of all stations (size = amenities, color = core/peripheral, ring = high unmet need)
- **Right**: Detailed station panel on click — demographics, amenity breakdown, equity metrics

Assumes the pipeline has already been run and `../data/processed/final_station_data.csv` exists.

In [2]:
# ── Install / upgrade visualization deps if needed ──────────────────────────
# Uncomment if running for the first time:
# !pip install folium plotly ipywidgets --quiet
!pip install "nbformat>=4.2.0"

In [3]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML
import ipywidgets as widgets

print("Dependencies loaded")

Dependencies loaded


## 1 · Load processed station data

In [4]:
# ── Load data ────────────────────────────────────────────────────────────────
DATA_PATH = "../data/processed/final_station_data.csv"

df = pd.read_csv(DATA_PATH)

# Normalise column names (pipeline may output slightly different names)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Ensure required columns exist with sensible defaults
required = {
    "station_name": "Unknown",
    "agency": "—",
    "station_type": "unknown",
    "latitude": None,
    "longitude": None,
    "total_amenities": 0,
    "grocery": 0,
    "park": 0,
    "clinic": 0,
    "pharmacy": 0,
    "childcare": 0,
    "ridership": 0,
    "median_income": np.nan,
    "pct_no_vehicle": np.nan,
    "pct_nonwhite": np.nan,
    "unmet_need_index": np.nan,
    "amenity_entropy": np.nan,
}
for col, default in required.items():
    if col not in df.columns:
        df[col] = default

df = df.dropna(subset=["latitude", "longitude"])
df["station_type"] = df["station_type"].str.lower().fillna("unknown")

core = df[df["station_type"] == "core"]
peri = df[df["station_type"] == "peripheral"]

print(f"Loaded {len(df)} stations  ({len(core)} core · {len(peri)} peripheral)")
df[["station_name", "agency", "station_type", "total_amenities", "unmet_need_index"]].head()

Loaded 79 stations  (40 core · 39 peripheral)


,station_name,agency,station_type,total_amenities,unmet_need_index
0,12th St. Oakland City Center,BART,core,53,0.024035
1,16th St. Mission,BART,core,45,0.058484
2,19th St. Oakland,BART,core,39,0.073065
3,24th St. Mission,BART,core,50,0.034129
4,Antioch,BART,peripheral,5,0.076270


## 2 · Summary statistics

In [5]:
def gini(arr):
    """Gini coefficient — 0 = equality, 1 = maximum inequality."""
    arr = np.sort(np.abs(arr[~np.isnan(arr)]))
    n = len(arr)
    if n == 0 or arr.sum() == 0:
        return np.nan
    return (2 * (np.arange(1, n + 1) * arr).sum()) / (n * arr.sum()) - (n + 1) / n

stats = pd.DataFrame({
    "group":            ["All", "Core", "Peripheral"],
    "n":               [len(df), len(core), len(peri)],
    "mean_amenities":  [df["total_amenities"].mean(), core["total_amenities"].mean(), peri["total_amenities"].mean()],
    "median_amenities":[df["total_amenities"].median(), core["total_amenities"].median(), peri["total_amenities"].median()],
    "gini":            [gini(df["total_amenities"].values), gini(core["total_amenities"].values), gini(peri["total_amenities"].values)],
    "mean_unmet_need": [df["unmet_need_index"].mean(), core["unmet_need_index"].mean(), peri["unmet_need_index"].mean()],
}).round(3)

display(stats.style.format({
    "mean_amenities": "{:.1f}", "median_amenities": "{:.1f}",
    "gini": "{:.3f}", "mean_unmet_need": "{:.3f}"
}).set_caption("Summary statistics by station type"))

,group,n,mean_amenities,median_amenities,gini,mean_unmet_need
0,All,79,15.9,11.0,0.428,0.210
1,Core,40,21.9,16.5,0.410,0.217
2,Peripheral,39,9.8,9.0,0.315,0.204


## 3 · Two-pane interactive visualization

**Left pane** — Plotly scatter-mapbox: every station as a bubble.  
Size encodes `total_amenities`. Color encodes `station_type`. Red ring marks high unmet need (top quartile).

**Right pane** — detail panel that updates when you click a station.

In [6]:
# ── Colour palette ───────────────────────────────────────────────────────────
COLORS = {
    "core":       "#378ADD",   # blue
    "peripheral": "#D85A30",   # coral
    "unknown":    "#888780",   # grey
    "unmet_ring": "#E24B4A",   # red
    "bg":         "#FAFAF8",
    "border":     "#D3D1C7",
}

# ── Marker sizes (scaled to amenity count) ───────────────────────────────────
a_min, a_max = df["total_amenities"].min(), df["total_amenities"].max()
SIZE_MIN, SIZE_MAX = 8, 28

def scale_size(val):
    if a_max == a_min:
        return (SIZE_MIN + SIZE_MAX) / 2
    return SIZE_MIN + (val - a_min) / (a_max - a_min) * (SIZE_MAX - SIZE_MIN)

df["_size"] = df["total_amenities"].apply(scale_size)
df["_color"] = df["station_type"].map(COLORS).fillna(COLORS["unknown"])

# Unmet need threshold (top 25 %)
unmet_thresh = df["unmet_need_index"].quantile(0.75)
df["_high_unmet"] = df["unmet_need_index"] >= unmet_thresh

print(f"Unmet need threshold (top 25%): {unmet_thresh:.3f}")
print(f"High-unmet-need stations: {df['_high_unmet'].sum()}")

Unmet need threshold (top 25%): 0.332
High-unmet-need stations: 20


In [7]:
# ── Build Plotly map figure ───────────────────────────────────────────────────
def build_map_figure(filter_type="all"):
    filtered = df if filter_type == "all" else df[df["station_type"] == filter_type]

    fig = go.Figure()

    # Layer 1 – unmet-need rings (drawn first so main dots sit on top)
    hi = filtered[filtered["_high_unmet"]]
    if len(hi):
        fig.add_trace(go.Scattermapbox(
            lat=hi["latitude"], lon=hi["longitude"],
            mode="markers",
            marker=dict(
                size=hi["_size"] + 10,
                color="rgba(226,75,74,0.25)",
                sizemode="diameter",
            ),
            hoverinfo="skip",
            name="High unmet need",
            showlegend=True,
        ))

    # Layer 2 – main station dots, one trace per type for clean legend
    for stype, label, color in [
        ("core",       "Core",       COLORS["core"]),
        ("peripheral", "Peripheral", COLORS["peripheral"]),
        ("unknown",    "Unknown",    COLORS["unknown"]),
    ]:
        sub = filtered[filtered["station_type"] == stype]
        if sub.empty:
            continue

        hover = (
            "<b>" + sub["station_name"] + "</b><br>"
            + sub["agency"] + " · " + sub["station_type"].str.capitalize() + "<br>"
            + "Amenities: " + sub["total_amenities"].astype(int).astype(str) + "<br>"
            + "Ridership: " + sub["ridership"].apply(lambda v: f"{v:,.0f}" if pd.notna(v) else "—") + "<br>"
            + "Unmet need: " + sub["unmet_need_index"].apply(lambda v: f"{v:.3f}" if pd.notna(v) else "—")
            + "<extra></extra>"
        )

        fig.add_trace(go.Scattermapbox(
            lat=sub["latitude"], lon=sub["longitude"],
            mode="markers",
            marker=dict(size=sub["_size"], color=color, sizemode="diameter"),
            text=sub["station_name"],
            customdata=sub.index.tolist(),
            hovertemplate=hover,
            name=label,
        ))

    center_lat = filtered["latitude"].mean() if len(filtered) else 37.75
    center_lon = filtered["longitude"].mean() if len(filtered) else -122.25

    fig.update_layout(
        mapbox=dict(
            style="carto-positron",
            center=dict(lat=center_lat, lon=center_lon),
            zoom=9.5,
        ),
        margin=dict(l=0, r=0, t=0, b=0),
        height=580,
        legend=dict(
            x=0.01, y=0.99,
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="#D3D1C7",
            borderwidth=0.5,
            font=dict(size=12),
        ),
        paper_bgcolor="white",
        clickmode="event+select",
    )
    return fig

print("Map builder ready")

Map builder ready


In [8]:
# ── Detail panel HTML generator ───────────────────────────────────────────────
def amenity_bar_html(label, val, max_val, color):
    pct = int(min(val / max(max_val, 1), 1) * 100)
    return (
        f'<div style="margin:4px 0;">'
        f'<div style="display:flex;justify-content:space-between;font-size:12px;margin-bottom:2px;">'
        f'<span style="color:#5F5E5A;">{label}</span>'
        f'<span style="font-weight:500;color:#2C2C2A;">{int(val)}</span></div>'
        f'<div style="background:#F1EFE8;border-radius:3px;height:6px;">'
        f'<div style="background:{color};width:{pct}%;height:6px;border-radius:3px;"></div>'
        f'</div></div>'
    )

def stat_row(label, value):
    return (
        f'<div style="display:flex;justify-content:space-between;'
        f'padding:5px 0;border-bottom:0.5px solid #F1EFE8;font-size:12px;">'
        f'<span style="color:#888780;">{label}</span>'
        f'<span style="font-weight:500;color:#2C2C2A;">{value}</span></div>'
    )

def build_detail_html(row):
    stype      = str(row.get("station_type", "unknown")).capitalize()
    color      = COLORS.get(str(row.get("station_type","unknown")).lower(), COLORS["unknown"])
    name       = row.get("station_name", "—")
    agency     = row.get("agency", "—")
    ridership  = row.get("ridership", np.nan)
    income     = row.get("median_income", np.nan)
    no_veh     = row.get("pct_no_vehicle", np.nan)
    nonwhite   = row.get("pct_nonwhite", np.nan)
    unmet      = row.get("unmet_need_index", np.nan)
    entropy    = row.get("amenity_entropy", np.nan)
    total_am   = row.get("total_amenities", 0)

    max_am = df["total_amenities"].max()
    amenity_cats = [("Grocery", "grocery", "#378ADD"),
                    ("Parks",   "park",    "#1D9E75"),
                    ("Clinics", "clinic",  "#D85A30"),
                    ("Pharma",  "pharmacy","#7F77DD"),
                    ("Childcare","childcare","#EF9F27")]

    bars = "".join(
        amenity_bar_html(lbl, row.get(col, 0), df[col].max() if col in df else 1, clr)
        for lbl, col, clr in amenity_cats
    )

    def fmt(v, fmt_str, suffix=""):
        return (fmt_str.format(v) + suffix) if pd.notna(v) else "—"

    # Unmet need colour indicator
    if pd.notna(unmet):
        unmet_color = "#E24B4A" if unmet >= unmet_thresh else "#1D9E75"
        unmet_label = f'<span style="color:{unmet_color};font-weight:500;">{unmet:.3f}</span>'
    else:
        unmet_label = "—"

    html = f"""
    <div style="font-family:system-ui,sans-serif;">
      <div style="display:flex;align-items:flex-start;gap:10px;margin-bottom:12px;">
        <div style="width:10px;height:10px;border-radius:50%;background:{color};margin-top:4px;flex-shrink:0;"></div>
        <div>
          <div style="font-size:15px;font-weight:500;color:#2C2C2A;line-height:1.3;">{name}</div>
          <div style="font-size:12px;color:#888780;margin-top:2px;">{agency} · {stype}</div>
        </div>
      </div>

      <div style="background:#F1EFE8;border-radius:8px;padding:10px 12px;margin-bottom:12px;">
        <div style="font-size:11px;color:#888780;margin-bottom:2px;">Total amenities within ½ mile</div>
        <div style="font-size:26px;font-weight:500;color:#2C2C2A;">{int(total_am)}</div>
      </div>

      <div style="margin-bottom:12px;">
        <div style="font-size:11px;font-weight:500;color:#5F5E5A;margin-bottom:6px;">Amenity breakdown</div>
        {bars}
      </div>

      <div style="margin-bottom:12px;">
        <div style="font-size:11px;font-weight:500;color:#5F5E5A;margin-bottom:4px;">Demographics</div>
        {stat_row("Median household income", fmt(income, "${:,.0f}"))}
        {stat_row("% No vehicle", fmt(no_veh, "{:.1f}", "%"))}
        {stat_row("% Non-white", fmt(nonwhite, "{:.1f}", "%"))}
        {stat_row("Avg weekday ridership", fmt(ridership, "{:,.0f}"))}
      </div>

      <div>
        <div style="font-size:11px;font-weight:500;color:#5F5E5A;margin-bottom:4px;">Equity metrics</div>
        {stat_row("Unmet need index", unmet_label)}
        {stat_row("Amenity entropy", fmt(entropy, "{:.3f}"))}
      </div>
    </div>
    """
    return html

print("Detail panel builder ready")

Detail panel builder ready


In [ ]:
# ── Two-pane interactive widget (no anywidget/FigureWidget needed) ───────────
# Uses a Dropdown to select stations + ipywidgets Output for the detail panel.
# The map auto-highlights the selected station.

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# -- Filter toggle
filter_buttons = widgets.ToggleButtons(
    options=[("All stations", "all"), ("Core only", "core"), ("Peripheral only", "peripheral")],
    value="all",
    style={"button_width": "140px"},
    layout=widgets.Layout(margin="0 0 8px 0"),
)

# -- Station dropdown (for detail panel)
station_names = sorted(df["station_name"].tolist())
station_dropdown = widgets.Dropdown(
    options=[(n, i) for i, n in zip(df.index, df["station_name"])],
    description="Station:",
    layout=widgets.Layout(width="340px"),
    style={"description_width": "60px"},
)

# -- Output areas
map_out    = widgets.Output()
detail_out = widgets.Output(layout=widgets.Layout(
    width="310px", min_height="560px",
    border="0.5px solid #D3D1C7",
    padding="0px", overflow_y="auto"
))

def render_map(filter_type="all"):
    map_out.clear_output(wait=True)
    with map_out:
        fig = build_map_figure(filter_type)
        # Highlight selected station
        sel_idx = station_dropdown.value
        if sel_idx is not None and sel_idx in df.index:
            row = df.loc[sel_idx]
            fig.add_trace(go.Scattermapbox(
                lat=[row["latitude"]], lon=[row["longitude"]],
                mode="markers",
                marker=dict(size=row["_size"] + 14, color="rgba(255,215,0,0.55)", sizemode="diameter"),
                hoverinfo="skip", showlegend=False, name="_selected",
            ))
        fig.show()

def render_detail(df_idx):
    detail_out.clear_output(wait=True)
    with detail_out:
        if df_idx is None or df_idx not in df.index:
            display(widgets.HTML(
                '<div style="color:#888780;font-size:13px;padding:20px;text-align:center;">'
                'Select a station above</div>'
            ))
            return
        row = df.loc[df_idx]
        display(widgets.HTML(build_detail_html(row)))

def on_filter_change(change):
    render_map(change["new"])

def on_station_change(change):
    render_map(filter_buttons.value)
    render_detail(change["new"])

filter_buttons.observe(on_filter_change, names="value")
station_dropdown.observe(on_station_change, names="value")

# -- Initial render
render_map("all")
render_detail(df.index[0])
station_dropdown.value = df.index[0]

header = widgets.HTML(
    '<h3 style="margin:0 0 4px;font-size:16px;font-weight:500;">'
    'Bay Area Transit Equity · Station Amenity Map</h3>'
    '<p style="margin:0 0 10px;font-size:12px;color:#888780;">'
    'Bubble size = total amenities &nbsp;|&nbsp; '
    '<span style="color:#378ADD;">●</span> Core &nbsp;'
    '<span style="color:#D85A30;">●</span> Peripheral &nbsp;'
    '<span style="color:#E24B4A;">○</span> High unmet need</p>'
)

two_pane = widgets.HBox(
    [map_out, detail_out],
    layout=widgets.Layout(gap="0px", align_items="flex-start", width="100%")
)

display(widgets.VBox([
    header,
    filter_buttons,
    station_dropdown,
    two_pane,
]))


## 4 · Supporting charts

Supplementary static charts for the report.

In [10]:
# ── 4a: Amenity distribution by station type ─────────────────────────────────
fig_dist = go.Figure()

for stype, color, name in [
    ("core", COLORS["core"], "Core"),
    ("peripheral", COLORS["peripheral"], "Peripheral"),
]:
    sub = df[df["station_type"] == stype]["total_amenities"].dropna()
    fig_dist.add_trace(go.Box(
        y=sub, name=name,
        marker_color=color,
        boxpoints="all", jitter=0.3, pointpos=-1.8,
        marker=dict(size=5, opacity=0.6),
    ))

fig_dist.update_layout(
    title="Total amenities: core vs peripheral",
    yaxis_title="Total amenities (½-mile radius)",
    height=420, width=560,
    paper_bgcolor="white", plot_bgcolor="white",
    yaxis=dict(gridcolor="#F1EFE8"),
    showlegend=False,
    margin=dict(l=50, r=20, t=50, b=40),
)
fig_dist.show()

In [11]:
# ── 4b: Unmet need index — top 15 stations ───────────────────────────────────
top15 = df.nlargest(15, "unmet_need_index")[["station_name", "station_type", "unmet_need_index"]].copy()
top15["color"] = top15["station_type"].map(COLORS).fillna(COLORS["unknown"])
top15 = top15.sort_values("unmet_need_index")

fig_unmet = go.Figure(go.Bar(
    x=top15["unmet_need_index"],
    y=top15["station_name"],
    orientation="h",
    marker_color=top15["color"],
    text=top15["unmet_need_index"].round(3),
    textposition="outside",
))
fig_unmet.update_layout(
    title="Top 15 stations by unmet need index",
    xaxis_title="Unmet need index",
    height=480, width=640,
    paper_bgcolor="white", plot_bgcolor="white",
    xaxis=dict(gridcolor="#F1EFE8"),
    margin=dict(l=160, r=60, t=50, b=40),
)
fig_unmet.show()

In [13]:
# ── 4d: Amenity category breakdown — grouped bar ─────────────────────────────
cats   = ["grocery", "park", "clinic", "pharmacy", "childcare"]
labels = ["Grocery", "Parks", "Clinics", "Pharmacy", "Childcare"]
colors_bar = ["#378ADD", "#1D9E75", "#D85A30", "#7F77DD", "#EF9F27"]

core_means = [core[c].mean() for c in cats]
peri_means = [peri[c].mean() for c in cats]

fig_bar = go.Figure([
    go.Bar(name="Core",       x=labels, y=core_means, marker_color=COLORS["core"]),
    go.Bar(name="Peripheral", x=labels, y=peri_means, marker_color=COLORS["peripheral"]),
])
fig_bar.update_layout(
    barmode="group",
    title="Mean amenity count by category (core vs peripheral)",
    yaxis_title="Mean count (½-mile radius)",
    height=420, width=620,
    paper_bgcolor="white", plot_bgcolor="white",
    yaxis=dict(gridcolor="#F1EFE8"),
    margin=dict(l=50, r=20, t=60, b=40),
)
fig_bar.show()

## 5 · Export all charts to PNG (optional)

In [ ]:
# Requires kaleido: pip install kaleido
# Uncomment to export:

# from pathlib import Path
# OUT = Path("../figures")
# OUT.mkdir(exist_ok=True)

# for name, fig in [
#     ("amenity_distribution", fig_dist),
#     ("unmet_need_top15",      fig_unmet),
#     ("novehicle_vs_amenities",fig_scatter),
#     ("category_breakdown",    fig_bar),
# ]:
#     fig.write_image(OUT / f"{name}.png", scale=2)
#     print(f"Saved {name}.png")

print("Export block ready — uncomment to run.")